# Instacart Pipeline — 5-Minute Deliverables Presentation

## Presentation goal

By the end of the presentation, the audience should understand how the team transformed raw Instacart files into analytics-ready Gold tables and how the repository proves that the results are reliable.

## Suggested timing

| Section | Time |
|---|---:|
| Opening and pipeline | 45 seconds |
| Gold model | 55 seconds |
| Validation | 45 seconds |
| Documentation and Git | 55 seconds |
| Dashboard and business questions | 60 seconds |
| Closing | 20 seconds |

Keep the explanation simple: show the evidence first, then explain why it matters.

## Slide 1 — We built an end-to-end analytics pipeline

### Show

- Repository README
- Project architecture image
- Medallion flow: **Source Files → Bronze → Silver → Gold → Analytics**

![Instacart pipeline architecture](https://raw.githubusercontent.com/ItsYangCoder/instacart-pipeline/main/src/images/Instacart_diagram.png)

### Say

“Our project transforms Instacart source files into reliable data for analysis. Bronze keeps the source-level data, Silver cleans and standardizes it, and Gold organizes it into a star schema for business questions.”

### Evidence

[README](https://github.com/ItsYangCoder/instacart-pipeline/blob/main/README.md) · [Architecture documentation](https://github.com/ItsYangCoder/instacart-pipeline/blob/main/docs/architecture_diagram.md)

## Slide 2 — Each layer has one clear responsibility

### Show

| Layer | What it does |
|---|---|
| Bronze | Ingests source files with minimal transformation |
| Silver | Cleans, standardizes, deduplicates, and integrates data |
| Gold | Creates business-ready dimensions and fact tables |
| Analytics | Answers business questions and creates useful charts |

### Say

“We separated the layers so each stage has a clear responsibility. This makes the pipeline easier to debug, validate, and maintain. Raw data is preserved for traceability, while business logic is prepared before it reaches Gold.”

### Evidence

[Bronze SQL files](https://github.com/ItsYangCoder/instacart-pipeline/tree/main/src/sql/01_bronze_ingest) · [Silver SQL files](https://github.com/ItsYangCoder/instacart-pipeline/tree/main/src/sql/02_silver_clean) · [Gold SQL files](https://github.com/ItsYangCoder/instacart-pipeline/tree/main/src/sql/03_gold_model)

## Slide 3 — Gold uses a simple star schema

### Show

**Fact table:** `fact_order_items`

- Grain: one row per product purchased within an order
- Primary key: `(order_id, product_id)`
- Measures/behavior fields: `add_to_cart_order`, `reordered`

**Dimension tables:**

- `dim_order` — one row per order
- `dim_products` — one row per product
- `dim_order_time` — one row per unique day-and-hour combination

### Say

“The fact table is the center because it records product purchases. The dimensions provide the context needed for analysis. The grain and keys are documented so everyone understands exactly what one row means.”

The fact table uses `timekey` to join to `dim_order_time.order_time_key`. This keeps the model consistent with the production table schema.

### Evidence

[Star schema documentation](https://github.com/ItsYangCoder/instacart-pipeline/blob/main/docs/star_schema.md) · [Data dictionary](https://github.com/ItsYangCoder/instacart-pipeline/blob/main/docs/data_dictionary.md)

## Slide 4 — Validation proves that the data is trustworthy

### Show

- Source, Silver, and Gold validation files
- The final release quality gate
- A green Databricks task or GitHub Actions result

### Explain the checks

- Row counts and non-empty tables
- Null checks on required fields
- Duplicate business-key checks
- Foreign-key checks between fact and dimensions
- Valid notebook, SQL, JSON, and job configuration files

### Say

“Validation is not just a final step. It checks whether the data is complete, unique, and connected to the correct dimensions. If the quality gate fails, the Databricks job and the deployment fail together.”

### Evidence

[Release quality gate](https://github.com/ItsYangCoder/instacart-pipeline/blob/main/tests/99_cicd_quality_gate.sql) · [All tests](https://github.com/ItsYangCoder/instacart-pipeline/tree/main/tests)

## Slide 5 — GitHub Actions makes the delivery repeatable

### Show

**Pull request flow:**

```
Pull Request → CI validation → Review and merge
```

**Branch flow:**

| Branch | Environment | Result |
|---|---|---|
| `develop` | Development | Test the pipeline safely |
| `main` | Production | Deploy the approved pipeline |

### What CI checks

- Code and file validity
- Secrets are not hardcoded
- Raw or oversized data files are blocked
- Destructive SQL is blocked
- Databricks task references are valid

### What CD does

After merge, GitHub Actions deploys the exact commit to Databricks, runs the pipeline in dependency order, and waits for the release quality gate.

### Say

“This makes deployment consistent. The same reviewed code is what gets tested and deployed. Development and production are separated, so we can test changes before releasing them.”

### Evidence

[CI/CD workflow](https://github.com/ItsYangCoder/instacart-pipeline/blob/main/.github/workflows/ci-cd.yml) · [Databricks job manifest](https://github.com/ItsYangCoder/instacart-pipeline/blob/main/ops/instacart_job.json) · [Successful production run](https://github.com/ItsYangCoder/instacart-pipeline/actions/runs/33891870676)

## Slide 6 — Gold tables answer business questions

### Show

Charts or outputs from the Business Questions notebook.

### Business questions

1. Which products and departments are purchased most frequently?
2. How does customer purchasing behavior change by day of week and hour of day?
3. Which products have the highest reorder behavior?
4. Which aisles have the highest reorder rate by day of the week?

### Say

“The Gold model is useful because it turns technical transformations into business answers. The fact and dimension tables can be reused for SQL analysis, charts, and dashboards.”

### Evidence

[Business Questions notebook](https://github.com/ItsYangCoder/instacart-pipeline/blob/main/src/sql/05_analytics/Business%20Questions.ipynb) · [Analytics SQL](https://github.com/ItsYangCoder/instacart-pipeline/blob/main/src/sql/05_analytics/business_views.sql)

## Slide 7 — Closing message

### Say

“Our deliverable is more than a collection of SQL files. It includes a traceable Bronze-to-Gold pipeline, a documented star schema, validation checks, meaningful Git contributions, analytics outputs, and automated development and production deployment.

The next production improvement would be true file-level incremental ingestion using a Delta control or watermark table. The current project already supports safe reruns and key-based `MERGE` behavior, but the source dataset itself is static.”

### Final checklist

- Pipeline: Bronze → Silver → Gold
- Model: facts, dimensions, grain, and keys
- Validation: checks and release quality gate
- Documentation: README, architecture, model, dictionary, assumptions
- Git: branches, pull requests, and meaningful commits
- Dashboard/analytics: charts answering business questions